# Advanced Modeling

Tujuan :

Setelah melalui tahap **Baseline Modeling**, **Historical Modeling**, dan **Elo Modeling**, proyek ini telah menghasilkan sekumpulan fitur yang semakin kaya dalam merepresentasikan kondisi masing-masing tim sebelum pertandingan berlangsung. Tahap berikutnya adalah mengevaluasi apakah algoritma yang lebih kompleks mampu memanfaatkan informasi tersebut secara lebih efektif dibandingkan model-model sebelumnya.

Pada notebook ini akan dilakukan eksperimen menggunakan dua algoritma *gradient boosting*, yaitu **XGBoost** dan **LightGBM**. Kedua algoritma dipilih karena dikenal memiliki performa yang baik pada berbagai permasalahan klasifikasi serta mampu menangkap hubungan non-linear antar fitur yang mungkin tidak dapat direpresentasikan secara optimal oleh model yang lebih sederhana. Agar hasil eksperimen dapat dibandingkan secara adil, seluruh konfigurasi utama akan dipertahankan sama dengan notebook pemodelan sebelumnya. Dataset yang digunakan tetap berasal dari hasil **Elo Feature Engineering**, proses pemisahan data tetap menggunakan **time-based train-test split**, dan seluruh preprocessing tetap dilakukan melalui **Pipeline** untuk menghindari *data leakage*.

Pada akhir notebook ini akan dilakukan perbandingan menyeluruh terhadap seluruh eksperimen pemodelan yang telah dilakukan, sehingga dapat ditentukan model terbaik yang akan digunakan sebagai dasar pembangunan **Match Prediction API** pada tahap berikutnya.

## 1. Load Elo Dataset

Memuat dataset hasil Elo Rating Engine yang akan digunakan sebagai dasar seluruh eksperimen Advanced Modeling. Karena kita ingin membandingkan algoritma, dataset yang dipakai harus sama dengan notebook sebelumnya.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/processed/elo_features.csv")

In [3]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_last5_winrate,away_last5_winrate,home_last10_winrate,...,away_avg_goals_scored,home_avg_goals_conceded,away_avg_goals_conceded,home_goal_difference_form,away_goal_difference_form,match_result,home_elo_before,away_elo_before,home_elo_after,away_elo_after
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,D,1500.000000,1500.000000,1500.000000,1500.000000
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,False,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,H,1500.000000,1500.000000,1510.000000,1490.000000
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,False,0.000000,0.500000,0.000000,...,2.000000,2.000000,1.000000,-1.000000,1.000000,H,1490.000000,1510.000000,1500.575011,1499.424989
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,False,0.333333,0.333333,0.333333,...,1.333333,1.333333,1.666667,0.333333,-0.333333,D,1499.424989,1500.575011,1499.458089,1500.541911
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,False,0.250000,0.250000,0.250000,...,1.750000,1.750000,1.500000,-0.250000,0.250000,H,1500.541911,1499.458089,1510.510716,1489.489284


In [4]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_last5_winrate,away_last5_winrate,home_last10_winrate,...,away_avg_goals_scored,home_avg_goals_conceded,away_avg_goals_conceded,home_goal_difference_form,away_goal_difference_form,match_result,home_elo_before,away_elo_before,home_elo_after,away_elo_after
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,D,1500.000000,1500.000000,1500.000000,1500.000000
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,False,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,H,1500.000000,1500.000000,1510.000000,1490.000000
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,False,0.000000,0.500000,0.000000,...,2.000000,2.000000,1.000000,-1.000000,1.000000,H,1490.000000,1510.000000,1500.575011,1499.424989
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,False,0.333333,0.333333,0.333333,...,1.333333,1.333333,1.666667,0.333333,-0.333333,D,1499.424989,1500.575011,1499.458089,1500.541911
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,False,0.250000,0.250000,0.250000,...,1.750000,1.750000,1.500000,-0.250000,0.250000,H,1500.541911,1499.458089,1510.510716,1489.489284


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49433 entries, 0 to 49432
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   date                       49433 non-null  str    
 1   home_team                  49433 non-null  str    
 2   away_team                  49433 non-null  str    
 3   home_score                 49433 non-null  float64
 4   away_score                 49433 non-null  float64
 5   tournament                 49433 non-null  str    
 6   neutral                    49433 non-null  bool   
 7   home_last5_winrate         49292 non-null  float64
 8   away_last5_winrate         49238 non-null  float64
 9   home_last10_winrate        49292 non-null  float64
 10  away_last10_winrate        49238 non-null  float64
 11  home_avg_goals_scored      49292 non-null  float64
 12  away_avg_goals_scored      49238 non-null  float64
 13  home_avg_goals_conceded    49292 non-null  float64
 14  a

Dataset yang digunakan merupakan hasil dari notebook Elo Rating Engine, yang telah menggabungkan tiga kelompok informasi utama:

- Baseline Features
- Historical Features
- Elo Rating

Dengan demikian, eksperimen pada notebook ini tidak lagi berfokus pada pembangunan fitur, melainkan mengevaluasi apakah algoritma Machine Learning yang lebih canggih mampu memanfaatkan informasi tersebut secara lebih efektif dibandingkan Logistic Regression dan Random Forest.